In [3]:
from sklearn.svm import SVR
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn import datasets

In [4]:
df = datasets.load_diabetes(as_frame=True).frame

In [5]:
df.head()
#  seems already scaled except target col

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [6]:
df.shape

(442, 11)

In [7]:
df.isnull().sum()

age       0
sex       0
bmi       0
bp        0
s1        0
s2        0
s3        0
s4        0
s5        0
s6        0
target    0
dtype: int64

In [8]:
X = df.drop("target", axis=1)
y = df["target"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [11]:
y.head()
# as to scale we need 2-d array 


0    151.0
1     75.0
2    141.0
3    206.0
4    135.0
Name: target, dtype: float64

In [16]:
y_scaler = StandardScaler()


y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1)).ravel()
# .values.reshape(-1, 1) make it 2-d
# .ravel() making it again 1-d 


In [17]:
model = SVR()

model.fit(X_train, y_train_scaled)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [18]:
y_test_pred_scaled = model.predict(X_test)
y_train_pred_scaled = model.predict(X_train)

In [20]:
print("train r2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2: ", r2_score(y_test_scaled, y_test_pred_scaled))
# 0.65 and 0.48 has gap too much hence its overfitting

train r2:  0.6596361676267712
test r2:  0.48844443151651884


In [21]:
# Linear
model = SVR(kernel="linear")

model.fit(X_train, y_train_scaled)

y_test_pred_scaled = model.predict(X_test)
y_train_pred_scaled = model.predict(X_train)

print("train r2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2: ", r2_score(y_test_scaled, y_test_pred_scaled))

train r2:  0.45191229982475245
test r2:  0.4433761323833776


In [22]:
# Linear
model = SVR(kernel="poly")

model.fit(X_train, y_train_scaled)

y_test_pred_scaled = model.predict(X_test)
y_train_pred_scaled = model.predict(X_train)

print("train r2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2: ", r2_score(y_test_scaled, y_test_pred_scaled))

train r2:  0.5790920834310542
test r2:  0.24203771038107758


# Hyperparameter tuning using GridSearchCV

In [23]:
from sklearn.model_selection import GridSearchCV

In [24]:
param_grid = {
    "C": [1, 2, 5, 10, 50, 100],
    "kernel": ["rbf", "linear"],
    "epsilon": [0.01, 0.1, 0.2, 0.3, 0.5]
}

In [25]:
svr_model = SVR()

grid_search = GridSearchCV(
    svr_model, 
    param_grid,
    scoring="r2", 
    cv=5
)

grid_search.fit(X_train, y_train_scaled)

,estimator,SVR()
,param_grid,"{'C': [1, 2, ...], 'epsilon': [0.01, 0.1, ...], 'kernel': ['rbf', 'linear']}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,kernel,'linear'


In [27]:
print("best params - ", grid_search.best_params_)

best params -  {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


In [28]:
best_model = SVR(kernel="linear", C=10, epsilon=0.1)

best_model.fit(X_train, y_train_scaled)

y_test_pred_scaled = best_model.predict(X_test)
y_train_pred_scaled = best_model.predict(X_train)

print("train r2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2: ", r2_score(y_test_scaled, y_test_pred_scaled))

train r2:  0.5151066486918876
test r2:  0.47444183250401095


In [29]:
# Linear svm same as kernel="linear" but not exactly same
from sklearn.svm import LinearSVR

model = LinearSVR(C=10, epsilon=0.1, max_iter=5000)

model.fit(X_train, y_train_scaled)

y_test_pred_scaled = model.predict(X_test)
y_train_pred_scaled = model.predict(X_train)

print("train r2: ", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2: ", r2_score(y_test_scaled, y_test_pred_scaled))

train r2:  0.5153185521125419
test r2:  0.4747626414198768
